# Spark - E-commerce Business Transaction

En esta sección se procesará el dataset de transacciones de comercio electrónico utilizando Spark.

## Componentes utilizados
Para el procesamiento del dataset se utilizó Apache Spark mediante PySpark, empleando principalmente el componente Spark SQL, que permite trabajar con datos estructurados mediante DataFrames y ejecutar operaciones de limpieza, transformación, filtrado, agrupación y agregación. Estas operaciones son ejecutadas por el motor Spark Core, encargado de gestionar las tareas, particiones y procesamiento distribuido de los datos. En este proyecto no se utilizaron Spark Streaming, MLlib ni GraphX, debido a que el análisis corresponde a un procesamiento por lotes de datos estructurados.

## Instalación de dependencias

In [65]:
!pip install -q pyspark

In [66]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType,StructField,StringType,DoubleType,IntegerType)

In [67]:
# Creamos la sesión
spark = (SparkSession.builder.appName("Ecommerce_Spark").getOrCreate())

print("Versión de Spark:", spark.version)

Versión de Spark: 4.0.4


## Descargamos el Dataset

In [68]:
import kagglehub
from pathlib import Path

DATASET_DIR = Path("/content/dataset")
LOCAL_FILE = DATASET_DIR / "sales_transaction.csv"

if not LOCAL_FILE.exists():
    cached = Path(kagglehub.dataset_download("gabrielramos87/an-online-shop-business"))

    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    LOCAL_FILE.write_bytes((cached / "Sales Transaction v.4a.csv").read_bytes())

    print(f"Dataset descargado en: {LOCAL_FILE}")

else:
    print(f"Dataset ya disponible: {LOCAL_FILE}")

Dataset ya disponible: /content/dataset/sales_transaction.csv


## Leemos el Dataset

In [69]:
df = (spark.read.option("header", True).option("inferSchema", True).option("nullValue", "NA").csv(str(LOCAL_FILE)))

# .option("nullValue", "NA") : Hara que cuando encuentre un valor "NA" en el CSV, deba interpretarlo como NULL.

print("Número de registros:", df.count())
print("Número de columnas:", len(df.columns))

Número de registros: 536350
Número de columnas: 8


## Realizamos una pequeña exploración inicial de los datos

In [70]:
# Visualizamos las columnas
print("Columnas:")
print(df.columns)

Columnas:
['TransactionNo', 'Date', 'ProductNo', 'ProductName', 'Price', 'Quantity', 'CustomerNo', 'Country']


In [71]:
# Tipos de datos
df.printSchema()

root
 |-- TransactionNo: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- ProductNo: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- CustomerNo: integer (nullable = true)
 |-- Country: string (nullable = true)



In [72]:
# Valores nulos
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)for c in df.columns]).show()

+-------------+----+---------+-----------+-----+--------+----------+-------+
|TransactionNo|Date|ProductNo|ProductName|Price|Quantity|CustomerNo|Country|
+-------------+----+---------+-----------+-----+--------+----------+-------+
|            0|   0|        0|          0|    0|       0|        55|      0|
+-------------+----+---------+-----------+-----+--------+----------+-------+



In [73]:
# Estadísticas
df.describe().show()

+-------+------------------+---------+------------------+--------------------+------------------+------------------+------------------+-----------+
|summary|     TransactionNo|     Date|         ProductNo|         ProductName|             Price|          Quantity|        CustomerNo|    Country|
+-------+------------------+---------+------------------+--------------------+------------------+------------------+------------------+-----------+
|  count|            536350|   536350|            536350|              536350|            536350|            536350|            536295|     536350|
|   mean| 559978.9296258752|     NULL|27501.334602703962|                NULL|12.662182287683445| 9.919347441036637|15227.893178194838|       NULL|
| stddev|13434.267079565068|     NULL|16621.547250098905|                NULL| 8.490450200816854|216.66229978946237|1716.5829320559192|       NULL|
|    min|            536365|1/10/2019|             10002|"Assorted Flower ...|              5.13|            -80

## Consultas

### Consulta 1 : Identificación y eliminación de valores nulos
Rubro : Limpieza de datos

In [74]:
# Filtramos
null_customer = df.filter(F.col("CustomerNo").isNull()).count()

print(f"Registros con CustomerNo nulo: {null_customer:,}")

Registros con CustomerNo nulo: 55


In [75]:
# Eliminamos
df = df.filter(F.col("CustomerNo").isNotNull())

In [76]:
# Verificamos
print("Registros después de eliminar CustomerNo nulo:",df.count())

Registros después de eliminar CustomerNo nulo: 536295


### Consulta 2 : Eliminación de duplicados
Rubro : Deduplicación

In [77]:
filas_antes = df.count()

df = df.dropDuplicates()

filas_despues = df.count()

duplicados_eliminados = filas_antes - filas_despues

print(f"Registros antes: {filas_antes:,}")
print(f"Duplicados eliminados: {duplicados_eliminados:,}")
print(f"Registros después: {filas_despues:,}")

Registros antes: 536,295
Duplicados eliminados: 5,200
Registros después: 531,095


### Consulta 3 : Transformación de fechas
Rubro: Transformación de variables

In [78]:
df = df.withColumn("Date",F.to_date(F.try_to_timestamp(F.col("Date").cast("string"),F.lit("M/d/yyyy"))))

In [79]:
df = (df.withColumn("Año", F.year("Date")).withColumn("Mes", F.month("Date")).withColumn("Día", F.dayofmonth("Date")))

In [80]:
# Verificamos
df.select("Date", "Año", "Mes", "Día").show(10)

+----------+----+---+---+
|      Date| Año|Mes|Día|
+----------+----+---+---+
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
|2019-12-09|2019| 12|  9|
+----------+----+---+---+
only showing top 10 rows


In [81]:
# Comprobamos que ninguna fecha se haya perdido durante la transformación
df.select(
    F.count(F.when(F.col("Año").isNull(), 1)).alias("Año_nulos"),
    F.count(F.when(F.col("Mes").isNull(), 1)).alias("Mes_nulos"),
    F.count(F.when(F.col("Día").isNull(), 1)).alias("Día_nulos")
).show()

+---------+---------+---------+
|Año_nulos|Mes_nulos|Día_nulos|
+---------+---------+---------+
|        0|        0|        0|
+---------+---------+---------+



### Consulta 4 : Creamos TotalSales
Rubro : Transformación de variables

In [82]:
df = df.withColumn("TotalSales",F.col("Price") * F.col("Quantity"))

In [83]:
# Verificamos
df.select("Price","Quantity","TotalSales").show(10)

+-----+--------+------------------+
|Price|Quantity|        TotalSales|
+-----+--------+------------------+
|11.06|      20|221.20000000000002|
|14.61|       8|            116.88|
| 6.19|       2|             12.38|
| 6.19|       1|              6.19|
| 6.19|       1|              6.19|
| 6.19|       1|              6.19|
| 7.24|      24|            173.76|
| 6.19|      12|             74.28|
| 6.19|       2|             12.38|
| 6.19|       1|              6.19|
+-----+--------+------------------+
only showing top 10 rows


### Consulta 5 : Filtrado de transacciones válidas
Rubro : Filtrado

In [84]:
df = df.filter((F.col("Quantity") >= 0) &(~F.col("TransactionNo").startswith("C")))

In [85]:
# Verificamos
print("Registros después del filtrado:", df.count())

Registros después del filtrado: 522601


### Consulta 6 : Facturación mensual
Rubro : Agrupación y agregación

In [92]:
monthly_sales = (df.groupBy("Año", "Mes").agg(F.sum("TotalSales").alias("TotalSales")).orderBy("Año", "Mes"))
monthly_sales.show(20)

+----+---+------------------+
| Año|Mes|        TotalSales|
+----+---+------------------+
|2018| 12| 4397648.390000064|
|2019|  1| 4548423.470000077|
|2019|  2| 3327342.640000061|
|2019|  3| 4384669.820000075|
|2019|  4| 3579310.060000059|
|2019|  5| 4569952.210000025|
|2019|  6| 4486050.150000029|
|2019|  7| 4571494.880000064|
|2019|  8|4749801.2300000265|
|2019|  9| 6613772.789999865|
|2019| 10|  7212279.84999983|
|2019| 11| 7828489.529999269|
|2019| 12| 2512069.519999769|
+----+---+------------------+



### Consulta 7 : Ingresos por país
Rubro: Agregación

In [87]:
country_sales = (df.groupBy("Country").agg(F.sum("TotalSales").alias("TotalSales")).orderBy(F.desc("TotalSales")))

country_sales.show(20)

+---------------+-------------------+
|        Country|         TotalSales|
+---------------+-------------------+
| United Kingdom|5.234679559999283E7|
|    Netherlands| 2151553.5900000026|
|           EIRE| 1711819.3899999978|
|        Germany| 1369839.6200000003|
|         France| 1329903.3899999976|
|      Australia|  995414.0099999983|
|         Sweden|  401879.8899999999|
|    Switzerland|          361691.96|
|          Japan|  293155.4399999998|
|          Spain|  280843.7999999997|
|        Belgium| 272131.88000000047|
|         Norway| 188612.52000000016|
|       Portugal| 175959.28000000003|
|        Finland| 120972.15000000002|
|        Denmark| 101083.98999999999|
|Channel Islands|  95932.24000000002|
|          Italy|  78536.24000000002|
|        Austria|  69147.26000000004|
|      Singapore|  63480.94999999999|
|         Cyprus| 61614.639999999956|
+---------------+-------------------+
only showing top 20 rows


### Consulta 8 : Top 10 productos por unidades vendidas
Rubro : Agrupación y ordenamiento

In [88]:
product_sales = (df.groupBy("ProductNo", "ProductName").agg(F.sum("Quantity").alias("TotalQuantity")).orderBy(F.desc("TotalQuantity")))

product_sales.show(10)

+---------+--------------------+-------------+
|ProductNo|         ProductName|TotalQuantity|
+---------+--------------------+-------------+
|    23843|Paper Craft Littl...|        80995|
|    23166|Medium Ceramic To...|        78033|
|    22197|      Popcorn Holder|        56902|
|    84077|World War 2 Glide...|        54951|
|   85099B|Jumbo Bag Red Ret...|        48375|
|   85123A|Cream Hanging Hea...|        37937|
|    21212|Pack Of 72 Retros...|        36492|
|    84879|Assorted Colour B...|        36394|
|    23084|  Rabbit Night Light|        30742|
|    22492|Mini Paint Set Vi...|        26633|
+---------+--------------------+-------------+
only showing top 10 rows


### Consulta 9 : Top 10 clientes por gastos
Rubro : Agrupación, agregación y ordenamiento

In [89]:
customer_sales = (df.groupBy("CustomerNo").agg(F.sum("TotalSales").alias("TotalSales")).orderBy(F.desc("TotalSales")))

customer_sales.show(10)

+----------+------------------+
|CustomerNo|        TotalSales|
+----------+------------------+
|     14646|2112282.0300000035|
|     16446|1002741.5700000001|
|     14911| 914204.1899999981|
|     12415| 900545.5399999997|
|     18102|         897137.36|
|     17450| 891069.5300000003|
|     12346|          840113.8|
|     14156| 694202.5100000005|
|     13694| 646116.7799999996|
|     17511| 639006.1900000002|
+----------+------------------+
only showing top 10 rows


### Consulta 10 : Estadísticos descriptivos
Rubro : Cálculo de métricas

In [90]:
df.select("Price","Quantity","TotalSales").summary("count","mean","stddev","min","max").show()

+-------+------------------+------------------+------------------+
|summary|             Price|          Quantity|        TotalSales|
+-------+------------------+------------------+------------------+
|  count|            522601|            522601|            522601|
|   mean|12.637160051353822|10.667492025464934|120.13238501264749|
| stddev| 7.965973735594675|157.54242013995727|1860.1586033147696|
|    min|              5.13|                 1|              5.13|
|    max|            660.62|             80995|1002718.1000000001|
+-------+------------------+------------------+------------------+



In [91]:
# Y para poder obtener la mediana
mediana = df.select(
    F.expr("percentile_approx(Price, 0.5)").alias("Price_median"),
    F.expr("percentile_approx(Quantity, 0.5)").alias("Quantity_median"),
    F.expr("percentile_approx(TotalSales, 0.5)").alias("TotalSales_median")
)

mediana.show()

+------------+---------------+-----------------+
|Price_median|Quantity_median|TotalSales_median|
+------------+---------------+-----------------+
|       11.94|              4|            44.48|
+------------+---------------+-----------------+

